In [ ]:
!pip install chromadb sentence_transformers groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import os
from groq import Groq



GROQ_API_KEY="YOUR_GROQ_API_KEY"

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)


In [ ]:
df=pd.read_csv('college_notes.csv')
print(df.shape, df.columns.tolist(),df.head(3))

(15, 4) ['note_id', 'subject', 'topic', 'content']   note_id  ...                                            content
0    N001  ...  ETL stands for Extract Transform Load. It is t...
1    N002  ...  A database is an organized collection of data ...
2    N003  ...  Data cleaning involves fixing or removing inco...

[3 rows x 4 columns]


In [ ]:
print(df['subjects'].value_counts())

In [ ]:
documents = df['content'].tolist()
metadatas = df[['subject', 'topic']].to_dict(orient='records')
ids = df['note_id'].astype(str).tolist()

print(f"Total chunks: {len(documents)}")
print(f"First doc ID: {ids[0]}")

Total chunks: 15
First doc ID: N001


In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes")

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Collection count: {collection.count()}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 97.9MiB/s]


Collection count: 15


In [ ]:
from chromadb.utils import embedding_functions

# Define the embedding function
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create/Reset the collection with the embedding function
# We use a new name or delete the old one to ensure the schema is updated
collection_with_emb = chroma_client.get_or_create_collection(
    name="college_notes_embedded",
    embedding_function=sentence_transformer_ef
)

# Add documents (embeddings will be generated automatically by the EF)
collection_with_emb.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"New collection count: {collection_with_emb.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


New collection count: 15


In [ ]:
# Update search function to use the new collection
def search_notes_v2(query, n_results=2):
    results = collection_with_emb.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test retrieval with embeddings
test_query = "What is ETL?"
res = search_notes_v2(test_query)
print(f"Retrieved: {res['documents'][0][0]}")

Retrieved: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


In [ ]:
def search_notes(query, n_results=2):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test search
query = "What is ETL?"
search_results = search_notes(query)
print(f"Search results for: {query}")
for doc in search_results['documents'][0]:
    print(f"- {doc}")

Search results for: What is ETL?
- ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
- An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.


In [ ]:
def generate_answer_v2(query):
    # Use the new collection with embeddings
    results = search_notes_v2(query, n_results=3)
    context = "\n".join(results['documents'][0])

    prompt = f"""You are a helpful college assistant. Use the following context to answer the student's question.\n\nContext:\n{context}\n\nQuestion: {query}"""

    completion = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return completion.choices[0].message.content

# Final test with embedding-based retrieval
question = "Explain ETL and how it relates to Data Engineering."
answer = generate_answer_v2(question)
print(f"Question: {question}\n")
print(f"AI Answer (Using Embeddings):\n{answer}")

Question: Explain ETL and how it relates to Data Engineering.

AI Answer (Using Embeddings):
I'd be happy to explain ETL and its connection to Data Engineering.

**What is ETL?**

ETL stands for Extract, Transform, Load. It's a process used to collect raw data from different sources, transform it into a clean and structured format, and load it into a database or data warehouse for analysis. The ETL process involves three main steps:

1. **Extract**: This step involves collecting raw data from various sources, such as databases, files, or external APIs. The goal is to gather all the necessary data required for analysis.
2. **Transform**: In this step, the extracted data is cleaned, processed, and transformed into a standardized format. This may involve data normalization, aggregation, or filtering.
3. **Load**: The final step involves loading the transformed data into a database or data warehouse, where it can be analyzed and queried.

**How does ETL relate to Data Engineering?**

Data 

In [ ]:
emb=SentenceTransformer("all-MiniLM-L6-v2")
print(emb.encode("hello").shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(384,)


In [ ]:
def retrieve_relevant_chunks(test_question, top_k=3):
    results = collection_with_emb.query(
        query_texts=[test_question],
        n_results=top_k
    )
    return results

# Run the function with a test question
test_question = "What is RAG?"
retrieval_results = retrieve_relevant_chunks(test_question, top_k=3)

print(f"Results for: {test_question}")
for i, doc in enumerate(retrieval_results['documents'][0]):
    print(f"Chunk {i+1}: {doc}")

Results for: What is RAG?
Chunk 1: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.
Chunk 2: Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.
Chunk 3: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
